# M4 — Prediction and walk-forward evaluation

Predict one-month total-return outperformance of URA. Compare simple baselines, Logistic Regression and Random Forest. Model choice uses the pre-2024 validation period only; settings and portfolio rules are fixed before the final holdout report.

Run cells in order with the project environment (`uv run jupyter lab`). Paths support the project root or `notebooks/`. See [workshop order](../_docs/workshop_order.md) for dependencies and checkpoints. Numerical outputs come from actual local inputs; no synthetic data are substituted for market results.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "config/project.yaml").exists():
    ROOT = ROOT.parent
assert (ROOT / "config/project.yaml").exists(), "Launch from the project root or notebooks/"
from uranium_signals.config import load_config
from uranium_signals.universe import load_universe
from uranium_signals.providers import acquire
from uranium_signals.run import research_tables
cfg = load_config(ROOT / "config/project.yaml")
universe = load_universe(cfg)
tickers = set(universe.ticker) | {cfg["benchmark"], cfg["broad_benchmark"]} | set(cfg["optional_benchmarks"])


In [ ]:
prices, macro, source_metadata = acquire(cfg, tickers, offline=True)
prices, quality, targets, table = research_tables(cfg, prices, macro, universe)


## 1. Inspect labels and future intervals

The primary label compares completed month-end adjusted-close returns. Missing future months remain missing. Execution returns are kept separately for the strategy simulation, since next-session open trades have a different interval.

In [ ]:
example = targets[targets.ticker.eq("CCJ")].tail(6)
display(example[["date", "label_end", "forward_return", "benchmark_forward_return", "outperform_1m", "execution_date", "exit_date", "trade_return"]])
known = table.dropna(subset=["outperform_1m"])
assert (known.label_end > known.date).all()
assert table.loc[table.label_end.isna(), "outperform_1m"].isna().all()


## 2. Expanding chronological fits

Fit once at the first monthly signal date of each calendar year, then predict that year's monthly cross-sections. Training labels must have ended strictly before the fit cutoff. Training-only median imputation and scaling live inside sklearn pipelines. There is no shuffled split. During the holdout, annual refits may use earlier holdout outcomes once known, as specified before evaluation.

In [ ]:
from uranium_signals.modeling import walk_forward, classification_metrics, select_model
predictions, fit_audit = walk_forward(table, cfg)
metrics, calibration = classification_metrics(predictions)
selected = select_model(metrics)
assert (fit_audit.latest_training_label_end < fit_audit.fit_cutoff).all()
display(fit_audit)
print("Model selected by validation balanced accuracy:", selected)
display(metrics[metrics.split.eq("validation")])


## 3. Evaluate the fixed holdout

Balanced accuracy weights both classes equally. Precision asks how often predicted outperformers outperform; recall asks how many outperformers were found. ROC AUC measures ranking, while PR AUC summarizes the precision-recall curve. Calibration compares predicted probabilities with observed frequencies. Results below must not be used to retune settings.

In [ ]:
display(metrics[metrics.split.eq("holdout")])
display(calibration[(calibration.model == selected) & (calibration.split == "holdout")])
holdout_calibration = calibration[(calibration.model == selected) & (calibration.split == "holdout")]
plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.scatter(holdout_calibration.mean_probability, holdout_calibration.observed_frequency)
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed frequency"); plt.title("Holdout probability bins")
plt.show()


## 4. Inspect model interpretation

A standardized logistic coefficient or tree importance is descriptive. Correlated features can split importance; these values do not identify economic causes. The latest signal model is trained only on completed labels, with the chosen model family unchanged.

In [ ]:
from uranium_signals.modeling import latest_signals
from uranium_signals.features import FEATURE_COLUMNS
signals, fitted_model = latest_signals(table, cfg, selected)
last_estimator = fitted_model.steps[-1][1]
values = last_estimator.coef_[0] if hasattr(last_estimator, "coef_") else last_estimator.feature_importances_
importance = pd.Series(values, index=FEATURE_COLUMNS).sort_values()
display(importance.rename("coefficient_or_importance"))
print("Final holdout evaluated; do not tune on this result.")


## Written-result checkpoint

Record whether either ML model improves on majority and momentum baselines, whether probabilities appear calibrated, and the main failure modes. Explain why prediction quality alone does not establish a profitable strategy.